In [ ]:
# ==============================================================================
# PROJETO: Análise Exploratória de Dados (EDA)
# CONTEXTO: Impacto do Saneamento na Mortalidade Infantil (Bahia)
# DISCIPLINA: Análise de Dados e Big Data (AV3)
# 
# NOTA DE TRATAMENTO CRÍTICO APLICADO NESTA VERSÃO:
# 1. Uso de imputações estatísticas e NaNs (vazio real) em vez de valores sentinela (-1.0).
# 2. Filtro do horizonte temporal (2018-2023) omitindo os anos de 2020 e 2021 
#    devido ao apagão de dados (Data Blackout) do SNIS gerado pela pandemia.
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo para os gráficos ficarem elegantes e padronizados
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ Bibliotecas carregadas e ambiente visual configurado!")


In [ ]:
# ==============================================================================
# FILTRO DA SÉRIE HISTÓRICA (Escopo do Professor + Correção da Pandemia)
# Mantemos apenas os anos válidos e reais solicitados, dropando o blackout de 2020/2021
# ==============================================================================

# Carregamento da base Gold consolidada estadual da Bahia
df_historico = pd.read_csv("data/gold/base_consolidada.csv")

anos_validos = [2018, 2019, 2022, 2023]

# Criando uma coluna de ano limpa caso ela venha compactada como co_anomes (ex: 201812 -> 2018)
if 'ano' not in df_historico.columns and 'co_anomes' in df_historico.columns:
    df_historico['ano'] = df_historico['co_anomes'].astype(str).str[:4].astype(int)

df_historico_filtrado = df_historico[df_historico['ano'].isin(anos_validos)].reset_index(drop=True)

print("📊 --- RESUMO ESTRUTURAL DA SÉRIE HISTÓRICA CORRIGIDA ---")
display(df_historico_filtrado.describe())


In [ ]:
# ==============================================================================
# PLOTAGEM DA EVOLUÇÃO TEMPORAL HISTÓRICA
# Gráfico de dois eixos para comparar Saneamento vs. Saúde Pública sem distorções
# ==============================================================================

fig, ax1 = plt.subplots(figsize=(10, 5))

# Eixo 1: Cobertura de Esgoto
color = '#2b7bba'
ax1.set_xlabel('Ano (Escopo Corrigido)', fontweight='bold')
ax1.set_ylabel('Taxa de Coleta de Esgoto (%)', color=color, fontweight='bold')
sns.lineplot(data=df_historico_filtrado, x='ano', y='tx_cobertura_esgoto', marker='o', color=color, ax=ax1, linewidth=2.5)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(anos_validos)

# Eixo 2: Mortalidade Infantil (Gêmeo do primeiro gráfico)
ax2 = ax1.twinx()  
color = '#e74c3c'
ax2.set_ylabel('Taxa de Mortalidade Infantil (por 1.000 nv)', color=color, fontweight='bold')
sns.lineplot(data=df_historico_filtrado, x='ano', y='taxa_mortalidade_infantil', marker='s', color=color, ax=ax2, linewidth=2.5)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Evolução Temporal: Saneamento vs. Mortalidade Infantil na Bahia\n(Nota: Anos de 2020/2021 omitidos devido ao apagão de dados do SNIS na pandemia)', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# ANÁLISE GRANULAR MUNICIPAL (Recorte Transversal 2022)
# Análise de Correlação com colunas padronizadas do pipeline Gold
# ==============================================================================

# Carregamento da base Gold consolidada municipal
df_consolidado = pd.read_csv("data/gold/base_consolidada_municipal_2022.csv")

# Como os valores sentinelas -1 foram removidos no pipeline de ETL (substituídos por mediana ou NaN),
# não precisamos mais filtrar '!= -1.0'. Basta usar .notna() para os nulos reais do Pandas.
df_municipal_filtrado = df_consolidado[
    df_consolidado['tx_atendimento_agua'].notna() & 
    df_consolidado['tx_coleta_esgoto'].notna() &
    df_consolidado['taxa_mortalidade_infantil'].notna()
].copy()

# Mapeamento para os novos nomes curtos e concisos das colunas
colunas_corr = [
    'taxa_mortalidade_infantil',
    'tx_atendimento_agua', 
    'tx_coleta_esgoto',
    'tx_coleta_lixo_pop_total'
]

colunas_validas = [c for c in colunas_corr if c in df_municipal_filtrado.columns]

plt.figure(figsize=(8, 6))
matriz_corr = df_municipal_filtrado[colunas_validas].corr()

# Plot do Heatmap focando na relação direta com a mortalidade infantil
sns.heatmap(matriz_corr[['taxa_mortalidade_infantil']], annot=True, cmap='RdBu_r', vmin=-1, vmax=1, fmt='.3f', linewidths=0.5)
plt.title('Matriz de Correlação de Pearson: Saneamento vs Mortalidade\n(Municípios Baianos - 2022)', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# NOTA DE INTERPRETAÇÃO CIENTÍFICA DO PARADOXO DA CORRELAÇÃO:
# Como observado acima, os coeficientes lineares puros ficam próximos de zero devido à:
# 1. Lei dos Pequenos Números: Cidades pequenas distorcem a taxa com poucos óbitos.
# 2. Multifatoriedade: Saúde imediata (leitos, vacinas) mascara o impacto de longo prazo.
#
# SOLUÇÃO APLICADA ABAIXO: Agrupamento por Faixas de Cobertura (Bins) para sumir com o ruído.
# ==============================================================================

# Ajustando as etiquetas caso o Pandas junte os blocos duplicados de valor 0.0
# Usamos duplicates='drop' para evitar o erro de bordas repetidas
df_municipal_filtrado['faixa_esgoto'] = pd.qcut(
    df_municipal_filtrado['tx_coleta_esgoto'], 
    q=3, 
    labels=['Baixa Cobertura', 'Alta Cobertura'] if df_municipal_filtrado['tx_coleta_esgoto'].quantile(1/3) == 0 else ['Baixa Cobertura', 'Média Cobertura', 'Alta Cobertura'],
    duplicates='drop'
)

# Calculando a taxa de mortalidade média para cada bloco de municípios
analise_faixas = df_municipal_filtrado.groupby('faixa_esgoto', observed=False)['taxa_mortalidade_infantil'].mean().reset_index()

# Plotando o gráfico de barras corrigido
plt.figure(figsize=(9, 5))
sns.barplot(data=analise_faixas, x='faixa_esgoto', y='taxa_mortalidade_infantil', palette='Blues_r')

# Adicionando os valores em cima das barras para facilitar a leitura visual
for index, row in analise_faixas.iterrows():
    plt.text(index, row['taxa_mortalidade_infantil'] + 0.2, f"{row['taxa_mortalidade_infantil']:.2f}", 
             color='black', ha="center", fontweight='bold')

plt.title('Abordagem Corrigida: Taxa de Mortalidade Infantil por Faixas de Coleta de Esgoto\n(Agrupamento de Municípios - 2022)', 
          fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Nível de Cobertura de Esgoto do Município', fontweight='bold')
plt.ylabel('Mortalidade Infantil Média (por 1.000 nv)', fontweight='bold')
plt.ylim(0, analise_faixas['taxa_mortalidade_infantil'].max() + 2)
plt.tight_layout()
plt.show()
